# **1.CZYSZCZENIE DANYCH**



In [ ]:
import pandas as pd

df = pd.read_csv("Online_Retail.csv", encoding='ISO-8859-1')

# usunięcie brakujących CustomerID
df = df.dropna(subset=["CustomerID"])

# usunięcie anulowanych transakcji (InvoiceNo zaczyna się na "C")
df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]

# usunięcie błędnych wartości
df = df[df["Quantity"] > 0]
df = df[df["UnitPrice"] > 0]

# konwersja daty
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

# usunięcie duplikatów
df = df.drop_duplicates()

# dodanie kolumny Revenue
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

df.head()

Dane zostały przygotowane poprzez usunięcie błędnych rekordów.

Usunięto brakujące wartości CustomerID, anulowane transakcje oraz rekordy z niepoprawnymi wartościami Quantity i UnitPrice.

Dodatkowo dokonano konwersji dat, usunięto duplikaty
oraz dodano kolumnę Revenue.

Czyszczenie danych jest kluczowe, ponieważ błędne dane prowadzą
do błędnych raportów i analiz.

# **2.WYBÓR ZIARNA**


Wybrano ziarno na poziomie pojedynczej pozycji faktury
(InvoiceNo + StockCode).

Jest to najczęściej stosowane podejście, ponieważ zapewnia
największą szczegółowość danych. Im bardziej szczegółowe ziarno,
tym większa elastyczność analiz, kosztem większego rozmiaru danych.

Taki wybór umożliwia analizę sprzedaży na poziomie produktów,
klientów oraz czasu, a także agregację danych do wyższych poziomów.

**Przykład analizy:**
Analiza sprzedaży konkretnych produktów w czasie oraz identyfikacja
najlepiej sprzedających się produktów.

# **3.MODEL GWIAZDY**

Zaprojektowano schemat gwiazdy składający się z jednej tabeli faktów (FactSales)
oraz trzech tabel wymiarów: DimCustomer, DimProduct i DimDate.

Tabela faktów przechowuje miary: Quantity oraz Revenue,
natomiast tabele wymiarów zawierają atrybuty opisowe.

Model gwiazdy został wybrany ze względu na prostotę oraz wydajność
w zapytaniach analitycznych. Umożliwia on łatwe łączenie danych
i szybkie generowanie raportów.

**DimCustomer**

In [ ]:
dim_customer = df[["CustomerID", "Country"]].drop_duplicates()
dim_customer = dim_customer.reset_index(drop=True)
dim_customer["CustomerKey"] = dim_customer.index + 1

dim_customer.head()

**DimProduct**

In [ ]:
dim_product = df[["StockCode", "Description"]].drop_duplicates()
dim_product = dim_product.reset_index(drop=True)
dim_product["ProductKey"] = dim_product.index + 1

dim_product.head()

**DimDate**

In [ ]:
dim_date = pd.DataFrame()
dim_date["InvoiceDate"] = df["InvoiceDate"].drop_duplicates()
dim_date = dim_date.sort_values("InvoiceDate").reset_index(drop=True)

dim_date["DateKey"] = dim_date.index + 1
dim_date["Year"] = dim_date["InvoiceDate"].dt.year
dim_date["Month"] = dim_date["InvoiceDate"].dt.month
dim_date["Day"] = dim_date["InvoiceDate"].dt.day

dim_date.head()

# **4.KLUCZE**

Zidentyfikowano klucze naturalne:
CustomerID dla klienta, StockCode dla produktu oraz InvoiceDate dla daty.

Wprowadzono klucze sztuczne:
CustomerKey, ProductKey oraz DateKey.

Każdy wymiar posiada własny klucz sztuczny, który jednoznacznie identyfikuje rekord.

Tabela faktów wykorzystuje wyłącznie klucze sztuczne,
co upraszcza relacje między tabelami oraz poprawia wydajność zapytań.

**Tabela faktów**

In [ ]:
fact = df.merge(dim_customer, on=["CustomerID", "Country"])
fact = fact.merge(dim_product, on=["StockCode", "Description"])
fact = fact.merge(dim_date, on="InvoiceDate")

fact_sales = fact[[
    "CustomerKey",
    "ProductKey",
    "DateKey",
    "Quantity",
    "Revenue"
]]

fact_sales.head()

# **5.SCD**

Zastosowano SCD typu 1 dla wymiaru klienta (DimCustomer).

Zmiany w atrybutach, takich jak Country, są nadpisywane,
co oznacza brak przechowywania historii zmian.

Rozwiązanie to jest proste w implementacji i wystarczające
dla podstawowych analiz danych.

In [ ]:
# przykładowa aktualizacja
dim_customer.loc[0, "Country"] = "NewCountry"

dim_customer.head()

# **6.Implementacja**

Implementacja została wykonana w języku Python z wykorzystaniem biblioteki Pandas.

Proces obejmował:
- wczytanie danych z pliku CSV,
- czyszczenie danych poprzez usunięcie błędnych rekordów,
- budowę tabel wymiarów,
- budowę tabeli faktów.

Dzięki zastosowaniu Pandas możliwe było efektywne przetwarzanie danych
oraz przygotowanie ich do modelu hurtowni danych w schemacie gwiazdy.

# ***Zadanie dodatkowe***

# **1.Rozszerzenie modelu**

**DimCountry**

In [ ]:
dim_country = df[["Country"]].drop_duplicates().reset_index(drop=True)
dim_country["CountryKey"] = dim_country.index + 1

dim_country.head()

Dodano nowy wymiar DimCountry, który zawiera unikalne kraje klientów.

Wymiar ten umożliwia analizę sprzedaży w ujęciu geograficznym,
np. porównanie sprzedaży pomiędzy krajami.

**Połączenie z fact**

In [ ]:
fact = fact.merge(dim_country, on="Country")

fact_sales = fact[[
    "CustomerKey",
    "ProductKey",
    "DateKey",
    "CountryKey",
    "Quantity",
    "Revenue"
]]

**Nowa miara**

In [ ]:
fact_sales["AvgPrice"] = fact_sales["Revenue"] / fact_sales["Quantity"]

Rozszerzono tabelę faktów o nową miarę AvgPrice,
która przedstawia średnią cenę sprzedaży produktu.

Pozwala to na bardziej szczegółową analizę sprzedaży oraz zmian cen.

**Denormalizacja**

Zastosowano podejście zdenormalizowane, charakterystyczne dla schematu gwiazdy.

Denormalizacja upraszcza zapytania oraz poprawia wydajność analiz,
ponieważ zmniejsza liczbę operacji łączenia tabel.

Wadą jest redundancja danych, jednak w hurtowniach danych
jest to akceptowalny kompromis.

# **2.SCD TYPE 2**

**przygotowanie kolumn**

In [ ]:
dim_customer_scd = dim_customer.copy()

dim_customer_scd["StartDate"] = pd.Timestamp("2020-01-01")
dim_customer_scd["EndDate"] = pd.NaT
dim_customer_scd["IsCurrent"] = 1

dim_customer_scd.head()

Zaimplementowano SCD typu 2 dla wymiaru klienta.

Dodano kolumny StartDate, EndDate oraz IsCurrent,
które pozwalają przechowywać historię zmian danych.

Każda zmiana atrybutu powoduje utworzenie nowego rekordu,
a poprzedni rekord zostaje oznaczony jako nieaktualny.

**symulacja zmiany**

In [ ]:
# wybierz przykładowego klienta
customer_id = dim_customer_scd.loc[0, "CustomerID"]

# stary rekord
old_record = dim_customer_scd[dim_customer_scd["CustomerID"] == customer_id].iloc[0]

# zamykamy stary rekord
dim_customer_scd.loc[
    dim_customer_scd["CustomerID"] == customer_id, "EndDate"
] = pd.Timestamp("2021-01-01")

dim_customer_scd.loc[
    dim_customer_scd["CustomerID"] == customer_id, "IsCurrent"
] = 0

# dodajemy nowy rekord (zmiana kraju)
new_record = old_record.copy()
new_record["Country"] = "NewCountry"
new_record["StartDate"] = pd.Timestamp("2021-01-01")
new_record["EndDate"] = pd.NaT
new_record["IsCurrent"] = 1

dim_customer_scd = pd.concat([dim_customer_scd, pd.DataFrame([new_record])], ignore_index=True)

dim_customer_scd.head()

W przypadku zmiany danych klienta,
poprzedni rekord zostaje zamknięty poprzez ustawienie daty EndDate
oraz oznaczenie go jako nieaktualny.

Następnie tworzony jest nowy rekord z aktualnymi danymi
i nową datą początkową.

Dzięki temu możliwe jest przechowywanie historii zmian
i analiza danych w czasie.

# **3.Pytania**

**Dlaczego wybrano takie ziarno?**

Wybrano ziarno na poziomie pojedynczej pozycji faktury,
ponieważ zapewnia najwyższą szczegółowość danych.
Pozwala to na analizę sprzedaży na poziomie produktów,
klientów oraz czasu, a także agregację danych do wyższych poziomów.

**Jakie kompromisy projektowe zastosowano?**

Zastosowano schemat gwiazdy, który jest podejściem zdenormalizowanym.
Pozwala on na szybsze i prostsze zapytania analityczne,
kosztem redundancji danych i większego rozmiaru tabel.

Dodatkowo zastosowanie szczegółowego ziarna zwiększa elastyczność analiz,
ale powoduje większą ilość danych do przetwarzania.

**Jak model wspiera analizy biznesowe?**

Model umożliwia analizę sprzedaży w różnych przekrojach,
takich jak klient, produkt, czas oraz kraj.

Dzięki temu możliwe jest identyfikowanie trendów sprzedaży,
najlepiej sprzedających się produktów oraz analiza zachowań klientów.

Zastosowanie schematu gwiazdy pozwala na szybkie generowanie raportów
i wspiera podejmowanie decyzji biznesowych.